In [ ]:
import torch
from torch import nn
from collections import OrderedDict
from torch.nn import functional as F


class Residual(nn.Module):  #@save
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=(1,1)):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=(3,1), padding=(1,0), stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=(3,1), padding=(1,0))
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,use_1x1conv=True, strides=(2,1)))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk

def Student():
    b1 = nn.Sequential(OrderedDict([
                    ('Conv',nn.Conv2d(1, 8, kernel_size=(3,2),stride = (1,2),padding = (1,0))),
                    ('BatchNorm2d',nn.BatchNorm2d(8)),
                    ('RelU',nn.ReLU()),
                    ('MaxPool',nn.MaxPool2d(kernel_size=(5,1), stride=(2,1),padding = (2,0)))
                    ]))
    b2 = nn.Sequential(*resnet_block(8, 16, 2))
    b3 = nn.Sequential(*resnet_block(16, 32, 2))
    b4 = nn.Sequential(*resnet_block(32, 64, 2))
    b5 = nn.Sequential(*resnet_block(64, 128, 2))
    net = nn.Sequential(b1, b2, b3, b4, b5,
                        nn.AdaptiveAvgPool2d((1, 1)),
                        nn.Flatten(), nn.Linear(128, 256),nn.Linear(256, 39))

    return net

In [1]:
import collections
import math
import os
import shutil
import pandas as pd
import torch 
import numpy as np
import torchvision
from torch import nn
from torch.utils.data import Dataset
from torch.nn import functional as F
from d2l import torch as d2l
from PIL import Image


In [2]:
class MyDataSet(Dataset):
    def __init__(self,folder_path):
        # 获取文件夹下所有文件的文件名列表
        all_files = os.listdir(folder_path)
        # 筛选出所有CSV文件
        csv_files = [f for f in all_files if f.endswith('.csv')]
        # CSV文件个数
        csv_num = (len(csv_files))
        data_total =[]
        data_total = np.array(data_total)
        for j in tqdm.tqdm(range(csv_num)):
            temp_path = csv_files[j]
            temp_csv_path = folder_path+"/"+ temp_path
            input_data = np.array(pd.read_csv(temp_csv_path))
            if j == 0:
                data_total = input_data
            else:
                data_total = np.vstack((data_total,input_data))
            #----
        self.data_total = data_total.astype(np.float32)
        valid_len = (self.data_total.shape[0] // 256) * 256
        self.data_total = self.data_total[:valid_len]
        
        self.data_total = self.data_total.reshape(-1, 256, self.data_total.shape[1])
        self.data_total[:,:,0] = self.data_total[:,:,0].clip(1, 39)
        self.len = self.data_total.shape[0]
        
        print('')
        print('===============')
        print(' ')
        print('输入数据总维度为 ', self.data_total[:,:,1:].shape)  # (N, 256, 8)
        print('label总维度为 ', self.data_total[:,:,0].shape)      # (N, 256)
        print(' ')
        print('===============')
        print('')
        
        
    def __len__(self):
        return self.len

    def __getitem__(self, index):
        features = torch.FloatTensor(self.data_total[index, :, 1:])  # (256, 8)
        label = torch.LongTensor([int(self.data_total[index, 0, 0])-1])  # 标签减1，使其范围变为0-38
        return features, label

In [ ]:
# Create dataset
# 实例化训练数据集
print("读取训练集数据，每个文件读{}条数据".format(singlefile_data_num))
train_dataset = MyDataSet(folder_path=train_path)
val_dataset = MyDataSet(folder_path=,singlefile_data_num=,aoa_path=)

# Create data loader with smaller batch size
train_iter  = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True,drop_last=True,num_workers=2,pin_memory=True)
valid_iter = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False,drop_last=True,num_workers=2,pin_memory=True)

SyntaxError: invalid syntax (4142714037.py, line 2)

# train

In [ ]:
student.train()

optimizer_stu = torch.optim.Adam(student.parameters(), lr=1e-4)
criterion_CE = nn.CrossEntropyLoss()
LOSS_KD = []

for epoch in range(20):
    for i, (data, target) in enumerate(train_iter):
        data, target = data.to(device), target.to(device)

        optimizer_cla.zero_grad()
        student_output = student(data)

        loss_CE = criterion_CE(student_output, target)
        acc = (student_output.argmax(1) == target).float().mean()
        LOSS_CE.append(loss_CE.item())
        ACC_CE.append(acc.item())

        loss_CE.backward()
        optimizer_cla.step()

        if i % 100 == 0:
            print('Epoch: {}, Iteration: {}, CE Loss: {}, Accuracy: {}'.format(epoch, i, loss_CE.item(), acc.item()))

# test

In [ ]:
from tqdm import tqdm

In [ ]:
student.eval()

valid_iter = tqdm(valid_iter, desc='Testing', ncols=120)
Length = 0
acc = 0
ACC_with_SNR = []
for i, (images, labels) in enumerate(valid_iter):
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():

        output = student(images)
       

        preds = output.argmax(dim=1)
        acc += (preds == labels).float().sum()

        Length = (i+1)*len(labels)

        valid_iter.set_postfix(Acc = acc.item() / Length)
        valid_iter.update()
    ACC_with_SNR.append(acc.item() / Length)

print(ACC_with_SNR)